# Markov-chain Monte Carlo simulations
---
REFS: 
- Boudreau, Applied Computational Physics
- Heath, Scientific Computing
- Landau y Paez, Computational physics, problem solving with computers
- Anagnostoupulos, A practical introduction to computational physics and scientific computing
- Ward, Numerical Mathematicas and Computing

  

The [Monte Carlo simulation method](https://en.wikipedia.org/wiki/Monte_Carlo_method) uses random sampling to perform computational experiments. One , very famous example, is the [Ising model (1920)](https://en.wikipedia.org/wiki/Ising_model), used to model magnetic materials (among many other applications). Here, each site on a discrete grid has a magnetic moment (or any other quantity) whose orientation (up/down, left/right, or angle) represents that site state, and there is an interaction with close neighbors (In 1D, neighbors left and right, in 2D, left/right/up/down). The interaction model can represent how the system respond to temperature and external magnetic field, or to external information pressure (for social systems) and so on. Here we will focus on the thermal 2D model that presents a phase transition. First, we will need to define some key concepts. The Ising model represents a system in the canonical ensemble (fixed temperature, number of particles and volume), where the samples are distributed according to a very specific distribution which you need to sample efficiently. To do so, you use a Markov-Chain and the Metropolis-Hastings algorithm.  The random process allows to simulate the intrinsic fluctuations when a system is in contact with a heat bath. 

In [ ]:
# Import needed utils, like for embeding videos and url
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'utils')))
from embed import embed


## Random systems, fluctuations, ensembles, ...
There are many systems where fluctations are intrinsic but they converge to a statistically stable final state, characterized by the avergae value of some observables and its fluctions . The later migh be functions of temperature, andl also the system size. 
### Ising model
```{figure} https://www.quantamagazine.org/wp-content/uploads/2020/06/ISING_2880x1620_Lede2.gif
:alt: 
:class: bg-primary
:width: 60%
:align: center
```
See <https://www.quantamagazine.org/the-cartoon-picture-of-magnets-that-has-transformed-science-20200624/>



In [ ]:
embed("https://mattbierbaum.github.io/ising.js/")

Applications to ML: <https://www.quantamagazine.org/the-strange-physics-that-gave-birth-to-ai-20250430/>

Other visualization tools:
- <https://physics.weber.edu/schroeder/software/demos/isingmodel.html>
- <https://www.veritasium.com/simulation4>

### A model gas
When a bunch of particles collide ellastically among them in a container, their speeds follow the [maxwell-boltzmann dstribution](https://en.wikipedia.org/wiki/Maxwell%E2%80%93Boltzmann_distribution?useskin=vector):

\begin{equation}
f(\mathbf{v})\, \mathrm{d}^3v = \left[ \frac{m}{2\pi k_B T}\right]^{3/2}
\exp\left(
-\frac{m v^2}{2k_B T}
\right)
\, \mathrm{d}^3v.
\end{equation}

or 
\begin{equation}
f(v)=\left(\frac{m}{2\pi k_B T}\right)^{3/2}
4\pi v^2
\exp\left(-\frac{m v^2}{2k_B T}\right).
\end{equation}

There is a constant momentun interchange among particles, even though the actual speed distribution remains constant (with fluctuations). 


In [ ]:
embed("https://phet.colorado.edu/sims/html/gas-properties/latest/gas-properties_all.html")

## Transision probabilities
Image a system which can be in any set of states. The rate at which the prob of being at state $i$ is equal to the prob of $j$ states going to $i$, minus the states leaving it, as expressed by the following  *master equation*

\begin{equation}
\frac{dw_i (t)}{dt} = \sum_j {w_j(t) R(j\to i) - w_i(t) R(i \to j)} = \sum_j {w_j(t) R_{ji} - w_i(t) R_{ij}},
\end{equation}

where $R(i \to j)$ is the transition probability from state $i$ to $j$, and $\sum_i w_i(t)= 1$. After a long time the system reaches a steady state  and $w_i(t)$ converge to finite numbers $p_i \le 0$. In that case one has 

\begin{equation}
0 = \sum_j \left [ {p_j R_{ji} - p_i R_{ij}} \right],
\end{equation}

### Detailed balance
A sufficient, although not necessary condition , to fullfil the previous conditions is called the *detailed balance* conditions, and it is to assume that the state is completely reversible:
\begin{equation}
p_j R_{ji} =  p_i R_{ij}.
\end{equation}

### Random sampling and the Markov chain appearance
It is important to note here that $p_i$ is somehow known, like in the canonical ensemble (N, V, T constant, E fluctuating), so the prob to be in a given state is
\begin{equation}
p_i = \frac{e^{-\beta E_i}}{Z},
\end{equation}
where $Z = \int dE e^{-\beta E}$ is the partition function. This is almost impossible to determine exactly for all systems, but yhe detailed balance condition helps since if gives the ratio among probabilities, so the constant and impossible factor $Z$ dissapears. From the detailed condition we have

\begin{equation}
\frac{R_{ij}}{R_{ji}} = \frac{p_j}{p_i}.
\end{equation}
So, if we devise an algorithm that generate samples like this, they fill **follow** the distribution $p$ and we will be able to model the random fluctuations and even compute observables as
\begin{equation}
\langle O \rangle = \frac{1}{N} \sum O_i,
\end{equation}
this is the so called importance sampling. 

Since the new state depends on the current state only, we also have here a markov chain. 


### Markov chain

SEE <https://www.youtube.com/watch?v=KZeIEiBrT_w>
- Free will
- Random processes in Nuclear tests
- Google page rank
- Text prediction

A [Markov chain](https://en.wikipedia.org/wiki/Markov_chain) is a sequence of states that are  produced by a stochastic process where the next state, $\hat X_{i+1}$ depends only on the current one, $\hat X_i$. **There is no memory**. The transition between this two states can be represented by the transition probability $p_{ij}$, which are the elements of and $M\times M$ matrix ($M$ the number of states). They fulfill $\sum_j p_{ij} = 1$. Here the transition matrix is independent on time.  Given two states, the *transition matrix*, $P_{ij}$ can be used to represent as,
\begin{equation}
\hat X_{j} = P \hat X_i.
\end{equation}
Using a super-script as denoting the iteration or algorithmic time, one has
\begin{equation}
\hat X^N = P \hat X^{N-1} = P^N \hat X^{0}.
\end{equation}
Under certain conditions, there is a final state independent on the initial condition (a fixed point), which sometimes is called as the ergodic property, 
\begin{equation}
\vec w = \lim_{N\to\infty} P^N \hat X^0.
\end{equation}
One way to ensure this is the so-called detailed balance condition
\begin{equation}
p_{ij} w_i = p_{ji} w_j.
\end{equation}
Summing over $j$ demonstrates that $\vec w$ is, indeed, a fixed point. By equating the fixed point with the desired distribution, by arranging the transition probabilities,  we complete the Markov chain procedure.

See <https://www.sharetechnote.com/html/WebProgramming/Websim_MarkovChain.html>

:::{exercise} Boudreau, 7.8.16 - Explicit transition matrix
Consider the following transition matrix
\begin{equation}
P = \begin{pmatrix}
3/4 & 1/4 & 0\\
0 & 2/3 & 1/3\\
1/4 & 1/4 & 1/2
\end{pmatrix}
\end{equation}
Find:
+ The eigen values and vectors
+ The left and right eigen vectors and values
+ The left and right fixed point probability vector, $\vec w$
+ $\lim_{n\to \infty} P^n$. Is the fixed point independent of the starting vector?
:::

## The Metropolis-Hastings algorithm

Refs:
- Metropolis, N., Rosenbluth, A. W., Rosenbluth, M. N., Teller, A. H., & Teller, E. (1953). Equation of state calculations by fast computing machines. The journal of chemical physics, 21(6), 1087-1092
- Hastings, W. K. (1970). Monte Carlo sampling methods using Markov chains and their applications.

The algorithm (Metropolis(1953) and Hastings(1970)) design appropriate transition probabilities to obtain a desired final distribution $p_i$, based on the previous result from detailed balance:

\begin{equation}
\frac{R_{ij}}{R_{ji}} = \frac{p_j}{p_i}.
\end{equation}

In this case, we want to devise a transition probability $R_{ij}$ that fulfills this. One can split it in a probability to generate the new state, and another to accept it

\begin{equation}
R_{ij}  = g_{ij} a_{ij}. 
\end{equation}

The original metropolis algorithm  Hastings extension is to not assume that $g_{ij}$ is reversible. There are several ways to fulfill the detailed balance conditions, as follows: 

- Metropolis-Hastings (<https://en.wikipedia.org/wiki/Metropolis%E2%80%93Hastings_algorithm?useskin=vector>):

\begin{equation}\label{eq:mh}
a_{ij} = \min\left(1, \frac{p_jg_{ji}}{p_ig_{ij}}\right).
\end{equation}

- Glauber, Heath-bath

\begin{equation}
a_{ij} = \frac{p_jg_{ji}} {p_ig_{ij}+p_jg_{ji}} = \frac{R_{ij}}{1+R_{ij}}.
\end{equation}

- Several others

In this case, we will use the Metropolis-Hastings proposal, to illustrate how to get samples following a given distribution and, finally, simulate the ising model.

How to use it?
- Generate a candidate new state with prob $g_{ij}$.
- Compute the acceptance probability $a_{ij}$
- Get a random number $r \in [0, 1)$. If $r < a_{ij}$, then accept the new state.

If $g_{ij} = g_{ji}$, we have the metropolis method (proposed by M(RT)^2).

Some possible problems to take into account:
- *Slow mixing*: not efficient exploration of isolated states.
- *Thermalization*: The simulation must run for a large time before reaching some kind of steady state. This depends, for instance, on temperature.
- *Autocorrelation*: Successive samples are correlated, so you need to generate a given number of samples before taking one into account if your ensemble. 
- *Multimodality* (many maxima/minima): To explore one region the system must pass on a low probability region. 

**Other sampling techniques**
- Cluster algorithms
- Guided random walks
- Microcanonical updating

### Example: Data from a given distribution
REF: <https://www.algorithm-archive.org/contents/metropolis/metropolis.html>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def target(x):
    return (
        10*np.exp(-4*(x+4)**2)
        +3*np.exp(-0.2*(x+1)**2)
        +np.exp(-2*(x-5)**2)
    )

def metropolis_step(x, sigma):
    """
    Performs one Metropolis update.
    """
    proposal = x + np.random.normal(scale=sigma) # Why? limits?
    alpha = min(1.0, target(proposal)/target(x)) # acceptance ratio
    if np.random.rand() < alpha:
        return proposal
    return x


def metropolis(nsteps, x0=0.0, sigma=1.0):
    """
    Generate samples using the Metropolis algorithm.
    """
    samples = np.empty(nsteps)
    x = x0
    for i in range(nsteps):
        x = metropolis_step(x, sigma)
        samples[i] = x

    return samples

# -------------------------------------------------------
# Generate samples
# -------------------------------------------------------
samples = metropolis(nsteps=50000, x0=0.0, sigma=1.0)

# -------------------------------------------------------
# Prepare figure
# -------------------------------------------------------
fig, ax = plt.subplots(figsize=(8,4))
x = np.linspace(-8,8,500)

# Scale the PDF to the histogram
pdf = target(x)
pdf /= np.trapezoid(pdf, x)

ax.set_xlim(-8,8)
ax.set_ylim(0,0.6)
line, = ax.plot(x,pdf,lw=2,label="Target")
hist = None

# -------------------------------------------------------
# Animation
# -------------------------------------------------------
def update(frame):
    global hist
    if hist is not None:
        for artist in hist[2]:
            artist.remove()
    n = (frame+1)*200 # Why?
    hist = ax.hist(samples[:n], bins=50, density=True, alpha=0.6, color='r') # Use same color always!
    ax.set_title(f"{n} samples")
    return hist[2] # why index 2?

ani = FuncAnimation(fig, update, frames=200, interval=50, blit=False)
plt.close()
HTML(ani.to_jshtml())

### Exercises

:::{exercise} Maxwell-Boltzman distribution
(Boudreau, 7.8.2) A gas is in thermal equilibrium at a temperature $T$, so its speed distribution is 
\begin{equation}
p(v) dv = \left( \frac{m}{2\pi \tau} \right)^{3/2} 4\pi v^2 e^{-mv^2/2\tau} dv,
\end{equation}
where $\tau = k T$, $m$ is the molecule mass, and $k_B$ is the Boltzmann constant. Non-dimensionalize this equation to show that this (Boltzmann) distribution can be sampled from the gamma distribution. Use Markov chain Monte Carlo.  Hint: Use $x = \sqrt{\frac{m}{2\tau}} v$ as the dimensionless variable. 
:::

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -------------------------------------------------------
# Physical parameters
# -------------------------------------------------------

m = 1.2323423
tau = 2.534534          # tau = k_B T

# -------------------------------------------------------
# Dimensionless Maxwell distribution
#
# p(x) ∝ x² exp(-x²),    x >= 0
# -------------------------------------------------------

def target(x):
    if x <= 0:
        return 0.0
    return x**2 * np.exp(-x**2)


# -------------------------------------------------------
# One Metropolis step
# -------------------------------------------------------

def metropolis_step(x, sigma):
    """
    Performs one Metropolis update.
    """
# YOUR CODE HERE
pass
    return x


# -------------------------------------------------------
# Generate dimensionless samples
# -------------------------------------------------------

def metropolis(nsteps, x0=1.0, sigma=0.8):
    samples = np.empty(nsteps)
    x = x0
    for i in range(nsteps):
        x = metropolis_step(x, sigma)
        samples[i] = x

    return samples

# -------------------------------------------------------
# Generate samples
# -------------------------------------------------------
x_samples = metropolis(nsteps=50000, x0=1.0, sigma=0.8)

# Remove burn-in
x_samples = x_samples[5000:]


# -------------------------------------------------------
# Transform back to the physical speed
#
# v = sqrt(2 tau / m) x
# -------------------------------------------------------
v_samples = np.sqrt(2 * tau / m) * x_samples

# -------------------------------------------------------
# Exact Maxwell speed distribution
# -------------------------------------------------------

def maxwell(v):
    return (
        (m / (2 * np.pi * tau))**1.5
        * 4 * np.pi * v**2
        * np.exp(-m * v**2 / (2 * tau))
    )


# -------------------------------------------------------
# Prepare figure
# -------------------------------------------------------
fig, ax = plt.subplots(figsize=(8,4))

v = np.linspace(0, 6, 500)

pdf = maxwell(v)

ax.set_xlim(0, 6)
ax.set_ylim(0, 1.1 * pdf.max())

ax.plot(v, pdf, lw=2, color='k', label='Exact distribution')

hist = None
hist = ax.hist(
    v_samples[:],
    bins=50,
    density=True,
    alpha=0.6,
    color='tab:blue'
)
plt.show()


:::{exercise} A discrete distribution
Using tha markov chain montecarlo, generate data following thediscrete distribution following $p_0 = 0.15, p_1 = 0.10, p_2 = 0.37, p_3 = 0.05, p_4 = 0.33$, where $p_i$ corresponds to the interval $[i, i+1)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -------------------------------------------------------
# Target probabilities
# -------------------------------------------------------

p = np.array([0.15, 0.10, 0.37, 0.05, 0.33])

# -------------------------------------------------------
# One Metropolis step
# -------------------------------------------------------

def metropolis_step(state):
# YOUR CODE HERE
pass

    return state


# -------------------------------------------------------
# Generate samples
# -------------------------------------------------------

def metropolis(nsteps, initial=0):

    samples = np.empty(nsteps, dtype=int)

    state = initial

    for i in range(nsteps):
        state = metropolis_step(state)
        samples[i] = state

    return samples


# -------------------------------------------------------
# Generate samples
# -------------------------------------------------------

samples = metropolis(50000)

# Discard burn-in
samples = samples[1000:]

# -------------------------------------------------------
# Plot
# -------------------------------------------------------

counts = np.bincount(samples, minlength=len(p))
counts = counts / counts.sum()

x = np.arange(len(p))

plt.figure(figsize=(6,4))

plt.bar(x-0.2, counts, width=0.4, label="Metropolis")
plt.bar(x+0.2, p,      width=0.4, label="Exact")

plt.xticks(x)
plt.xlabel("State")
plt.ylabel("Probability")
plt.legend()

plt.show()

## Statistical mechanics and observables
In the canonical ensemble the probability to sample a state $\mu$, with energy $E_\mu$, at temperature $T$, is given by 

\begin{equation}
p_\mu = \frac{1}{Z} e^{-\beta E_\mu},
\end{equation}

where $\beta = 1/kT$ and $Z = \sum_\mu e^{-\beta E_\mu}$ is called the partition function. Is not simply a normalization factor, it stores the information of the system. For instance, to compute an observable average $\langle O \rangle$, one has

\begin{equation}
\langle O \rangle = \sum_\mu p_\mu O_\mu.
\end{equation}

When we sample with the actual system distribution, as in the Metropolis-Hastings method, it is called **importance sampling** and the average value for a given observable is simply

\begin{equation}
\langle O \rangle = \frac{1}{N} \sum O_i,
\end{equation}
where the samples must be independent (not correlated).

For the mean energy, one has

\begin{equation}
\langle E \rangle = \sum_\mu p_\mu E_\mu = \sum_\mu \frac{1}{Z} e^{-\beta E_\mu} = -\frac{\partial \ln Z}{\partial \beta}
\end{equation}

The specific heat can be computed as 

\begin{equation}
C = \frac{\partial U}{\partial T} = k\beta^2 \frac{\partial^2 \ln Z}{\partial \beta^2} =  k\beta^2 (\Delta E) ^2.
\end{equation}

The entropy can be shown to be

\begin{equation}
S = -k\sum_\mu p_\mu \ln p_\mu
\end{equation}



## Ising model
- Ising, E. (1925), "Beitrag zur Theorie des Ferromagnetismus", Z. Phys., 31 (1): 253–258, doi:10.1007/BF02980577, S2CID 122157319

In the [Ising model](https://en.wikipedia.org/wiki/Ising_model?useskin=vector), each site on a grid is assigned a given state, up or down
```{figure} https://qph.cf2.quoracdn.net/main-qimg-0fb41bd632ba1eea8a654828dcff5d93-pjlq
:alt: 
:class: bg-primary
:width: 80%
:align: center
```

In [ ]:
embed("https://mattbierbaum.github.io/ising.js/")

### The model
```{figure} https://www.quantamagazine.org/wp-content/uploads/2020/06/ISING_2880x1620_Lede2.gif
:alt: 
:class: bg-primary
:width: 60%
:align: center
```

The state of cell $i$ will be denote as $\sigma_i = \pm 1$. The interaction with the neighbors are represented by the Hamiltonian 

\begin{equation}
H = -J \sum \sigma_i \sigma_{j} - B\sum\sigma_i,
\end{equation}

where $J$ is the coupling constant and the sum is over all $i$ sites and $j$ are all the neighbors of that site, and $B$ is the strength of an external magnetic field. If $J>0$ then the system is ferromagnetic (favors equally oriented spins). The difficulty of modelling a system like this is that a simple $5\times 5$ system, the partition function would need to be summed up on $2^25 \simeq 3.4\times 10^6$ terms.

One can take samples for observable $O$ after equilibrium has been reached, and the average is
\begin{equation}
\langle O \rangle = \frac{1}{n_s} \sum_s O_s,
\end{equation}
where $n_s$ is the number of independent monte carlo samples, taken in equilibrium, and $s$ denotes the particular state.

Reference observables of interest on the Ising model are, at a given state $s$,
- The internal energy
\begin{equation}
E_s = -J \sum \sigma_i \sigma_{j} - B\sum\sigma_i,
\end{equation}
- The magnetization:
  \begin{equation}
  M_s = N\langle m_s \rangle =  \sum \sigma_i
  \end{equation}  
- The specific heat
  \begin{equation}
  C = \frac{\partial \langle E \rangle }{\partial T} = T\frac{\partial S}{\partial T} = -\beta\frac{\partial S}{\partial \beta} =  k_B\beta^2(\langle E^2 \rangle - \langle E \rangle^2).
  \end{equation}
- The magnetic susceptibility:
  \begin{equation}
  \chi = \frac{\partial \langle M \rangle}{\partial B} =  \frac{\beta}{N}(\langle M^2 \rangle - \langle M \rangle^2).
  \end{equation}

### Some results at equilibrium

We would like to simulate the Ising system to reproduce an interesting phenomena: a phase transition. For $B=0$, the system will be magnetized (or not) and the behaviour will be clearly determined for a critical temperature 

\begin{equation}
\beta_c = \frac{1}{2}\ln(1+\sqrt{2}) \simeq  0.4406867935\ldots
\end{equation}

```{figure} https://miro.medium.com/max/1400/1*ODPEcoAf3Rwfyyo7lTsNFw.png
:alt: 
:class: bg-primary
:width: 80%
:align: center
From: https://towardsdatascience.com/monte-carlo-method-applied-on-a-2d-binary-alloy-using-an-ising-model-on-python-70afa03b172b

```

See also the following for some discussion about the phase transition: 
- <https://www.ippp.dur.ac.uk/~krauss/Lectures/NumericalMethods/PhaseTransitions/Lecture/pt1.html>
- <https://www.ryanirl.com/blog/phase-transitions-ising/>
- <https://www.researchgate.net/figure/The-finite-size-scaling-analysis-of-the-fidelity-susceptibility-for-the-period-2_fig1_364669709>
- <https://pyfssa.readthedocs.io/en/stable/index.html>
- <https://en.wikipedia.org/wiki/Universality_class?useskin=vector>



### Montecarlo simulation: the metropolis algorithm

Since the system is in the canonical ensemble (fixed temperature) it should be sampled from 

\begin{equation}
p(X) = \frac{e^{-\beta H(X)}}{Z},
\end{equation}

where $\beta = \frac{1}{k_B T}$ is the Boltzmann factor and $Z$ is the partition function, $Z = \sum e^{-\beta H(x)}$. Given that the Hamiltonian is local, applying the Metropolis algorithm to this canonical probability simplifies to computing just the change with interacting neighbors. To generate a new state, we will just choose a new site at random to flip its spin, so the generating prob, $g$, is symmetrical. Therefore, we reduce ourselves to the metropolis algorithm, which, combined with the canonical boltzmann distribution gives, for the acceptnce probability,
\begin{equation}
a_{ij} = \min \left(1, e^{-\beta(H(x') - H(x))} \right) = \min \left(1, e^{-\beta(E' - E)} \right) = \min \left(1, e^{-\beta \Delta E} \right).
\end{equation}

Therefore, you generate a new sample, if the energy is less than the previous one, the new sample is accepted. Otherwise, a random number $z \in [0, 1) $ is thrown and the new sample is accepted if $z < e^{-\beta \Delta E}$. $\Delta E$ can be easily computed for a 2D system: If a site $i$ is selected and flipped, and its original value is $\sigma_i$, then $\Delta E = E' - E = 2J\sigma_i \sum_j \sigma_j$, where the sum is performed over the neighbors.

**Simulation schematics**

We will use a periodic lattice of size $N\times N$ (Toroidal geometry). We will need  functions for
- Setting up the initial state
  + Input: The lattice
  + Output: The lattice with some initial condition
- Perform a montecarlo step:
  + Input: The lattice, the J constant value, a random number generator already seeded
  + Output: The lattice with the corresponding change. 
- Plot/visualize the system: 
  + Input: The lattice
  + Output: A graphical representation
- Observables: compute the energy, magnetization, specific heat, etc. 
  + Input: The lattice
  + Output: The observables values at that realization
- Perform the simulation: Iterate over time, performing many MonteCarlo steps. Also accumulates the observables

:::{note} Simulation details 
- If the system size is small, the finite size effects will be important
- For a given temperature, we need to simulate it during a long time, discarding the initial data (by how much?) , taking samples after equilibration, but independently (how?)
- To improve the data quality, we either use a very large system, or use several seeds and compute averages, or both.
:::

Based on : <https://rajeshrinet.github.io/blog/2014/ising-model/> and <https://medium.com/data-science/monte-carlo-method-applied-on-a-2d-binary-alloy-using-an-ising-model-on-python-70afa03b172b>a

In [ ]:
# Import cell
import numpy as np
import matplotlib.pyplot as plt
from numba import njit

In [ ]:
# functions before the main application

def initial_state(rng, n):
    """
    Generates a random spin configuration
    """
    return 2*rng.integers(0, 2, size=((n,n))) - 1 # 2*[0, 1) - 1 -> integers either -1 or 1  


def mc_single_step(rng, grid, beta):
    """
    Monte Carlo move using the metropolis algorithm
    Lattice of size nxn
    NOTE: Periodic neighbors as:
    right: (ii+1)%n
    left: (ii-1+n)%n
    Similar for up nd down
    """
    # YOUR CODE HERE
    pass
        

def mc_full_step(rng, grid, beta):
    """
    Perform a full (on average on every site) monte carlo step
    """
    n = grid.shape[0]
    n2 = n*n
    for istep in range(0, n2):
        mc_single_step(rng, grid, beta)

In [ ]:
## Observables

@njit
def compute_energy(grid):
    """
    Energy for this configuration
    """
    # YOUR CODE HERE
    pass

@njit
def compute_magnetization(grid):
    # YOUR CODE HERE
    pass
    

In [ ]:
def evolve_one_seed(seed, L, beta, ntotal):
    """
    Performs a full evolution and stores data
    """
    SEED = seed # can be itemp or whatever
    rng = np.random.default_rng(seed=SEED)

    Edata = np.zeros(ntotal)
    Mdata = np.zeros_like(Edata)
    
    # create the grid
    grid = initial_state(rng, L)

    # Sampling (what about correlation?)
    for isample in range(ntotal):
        mc_full_step(rng, grid, beta)
        energy = compute_energy(grid)
        magnetization = compute_magnetization(grid)
        Edata[isample] = energy
        Mdata[isample] = magnetization

    return Edata, Mdata, grid

def compute_observables(Evalues, Mvalues, L, beta):
    # Compute mean values
    e1 = np.sum(Evalues)
    e2 = np.sum(Evalues*Evalues)
    m1 = np.sum(Mvalues)
    m2 = np.sum(Mvalues*Mvalues)
    
    # Compute averages and also normalize by size
    nsample = len(Evalues)
    n1 = 1.0/(nsample*L*L)
    n2 = n1/nsample
    E = e1*n1
    M = m1*n1
    C = (n1*e2 - n2*e1*e1)*beta*beta
    X = (n1*m2 - n2*m1*m1)*beta

    return E, M, C, X

### Equilibration test for a single temperature and a single seed

In [ ]:
# Constants
L = 15 # grid size
TEMP = np.array([1.53])
NTEMP = len(TEMP) # Number of temperature points
BETA = 1.0/TEMP # KB=1
NEQ = 300 # Equilibration mcss steps (full sweep)
NSAMPLE = 1000 # mcss for samples
NTOTAL = NEQ + NSAMPLE

# Observables
Edata = np.zeros((NTEMP, NTOTAL))
Mdata = np.zeros_like(Edata)
Cdata = np.zeros_like(Edata)
Xdata = np.zeros_like(Edata)

In [ ]:
SEED = 42 # can be itemp or whatever
Edata[0], Mdata[0], _ = evolve_one_seed(SEED, L, BETA[0], NTOTAL)
E, M, C, X = compute_observables(Edata[0][NEQ:], Mdata[0][NEQ:], L, BETA[0])

In [ ]:
# Plot the time series
fig, ax = plt.subplots()
ax.plot(Edata[0])
ax.plot(Mdata[0], '-o')
ax.set_xlabel("mcss")
ax.set_ylabel("E, M")

### Evolution for several seeds, taking averages 


In [ ]:
fig, ax = plt.subplots()
for seed in [42, 43, 44, 45, 6554]:
    Edata[0], Mdata[0], _  = evolve_one_seed(seed, L, BETA[0], NTOTAL)
    E, M, C, X = compute_observables(Edata[0][NEQ:], Mdata[0][NEQ:], L, BETA[0])
    print(E, M, C, X)
    # Plot the time series
    ax.plot(Edata[0])
    ax.plot(Mdata[0], '-o')
        
ax.set_xlabel("mcss")
ax.set_ylabel("E, M")

In [ ]:
def evolve_several_seeds(seeds, beta, L, ntotal, neq):
    """
    Average the average values per time series
    """
    Etmp = np.zeros(len(seeds))
    Mtmp = np.zeros_like(Etmp)
    Ctmp = np.zeros_like(Etmp)
    Xtmp = np.zeros_like(Etmp)
    print("seed: ", end = " ")
    for iseed, seed in enumerate(seeds):
        print(f"{iseed+1}/{len(seeds)} ", end=" ")
        Edata, Mdata, last_grid = evolve_one_seed(seed, L, beta, ntotal)
        Etmp[iseed], Mtmp[iseed], Ctmp[iseed], Xtmp[iseed] = compute_observables(Edata[neq:], Mdata[neq:], L, beta)
    print("")
    return np.average(Etmp), np.average(Mtmp), np.average(Ctmp), np.average(Xtmp), np.std(Etmp), np.std(Mtmp), np.std(Ctmp), np.std(Xtmp), last_grid


In [ ]:
SEEDS = [42, 21, 45, 87]
E, M, C, X, sigmaE, sigmaM, sigmaC, sigmaX, _ = evolve_several_seeds(SEEDS, BETA[0], L, NTOTAL, NEQ)
print(E, M, C, X, sigmaE, sigmaM, sigmaC, sigmaX)


###  Evolution for several temperatures

In [ ]:
# Constants
L = 15
NTEMP = 10 # Number of temperature points
TEMP = np.linspace(1.53, 3.28, NTEMP)
BETA = 1.0/TEMP # KB=1

SEEDS = [42, 1, 10, 21]

# Plot settings for whole system matrix
NCOLS = 5
NROWS = int(NTEMP/NCOLS)

# Observables as function of temp
E, M, C, X = np.zeros(NTEMP), np.zeros(NTEMP), np.zeros(NTEMP), np.zeros(NTEMP)
sigmaE, sigmaM, sigmaC, sigmaX = np.zeros(NTEMP), np.zeros(NTEMP), np.zeros(NTEMP), np.zeros(NTEMP)

In [ ]:
fig, ax = plt.subplots(NROWS, NCOLS, sharex = True, sharey = True)

for itemp in range(NTEMP):
    print(f"itemp: {itemp+1}/{NTEMP}")
    E[itemp], M[itemp], C[itemp], X[itemp], sigmaE[itemp], sigmaM[itemp], sigmaC[itemp], sigmaX[itemp], last_grid = evolve_several_seeds(SEEDS, BETA[itemp], L, NTOTAL, NEQ)
    ax[itemp//NCOLS, itemp%NCOLS].imshow(last_grid)
    ax[itemp//NCOLS, itemp%NCOLS].set_title(rf"$T = {TEMP[itemp]:.2f}$")
plt.tight_layout()

In [ ]:
# Plot the observables
# Critical temp: 2.269
fig, ax = plt.subplots(2,2, sharex = True)
ax[0, 0].plot(TEMP, E, '-o')
ax[0, 0].set_title("Energy")
ax[0, 1].plot(TEMP, M, '-o')
ax[0, 1].set_title("Magnetization")
ax[1, 0].plot(TEMP, C, '-o')
ax[1, 0].set_title("Specific heat")
ax[1, 1].plot(TEMP, X, '-o')
ax[1, 1].set_title("Susceptibility")


## More topics:
- Percolation
- Fractals
- Scaling and critical exponents
- Universality and renormalization

## Exercises
- [ ] Plot the observable for several system sizes. Is there any dependence? Notice that you have to "undo" the normalization

In [ ]:
# Constants
LVALS = [5, 10, 15, 20]
NTEMP = 10 # Number of temperature points
TEMP = np.linspace(1.53, 3.28, NTEMP)
BETA = 1.0/TEMP # KB=1

SEEDS = [42, 1, 10, 21]

fig, ax = plt.subplots(2,2, sharex = True)
ax[0, 0].set_title("Energy")
ax[0, 1].set_title("Magnetization")
ax[1, 0].set_title("Specific heat")
ax[1, 1].set_title("Susceptibility")

# YOUR CODE HERE
pass

ax[0, 0].legend()
ax[0, 1].legend()
ax[1, 0].legend()
ax[1, 1].legend()

- [ ] Compute the same observable but average over several seeds to compute the mean and the error of the mean

In [ ]:
L = 15
NTEMP = 10 # Number of temperature points
TEMP = np.linspace(1.53, 3.28, NTEMP)
BETA = 1.0/TEMP # KB=1

SEEDS = [42, 1, 10, 21]

fig, ax = plt.subplots(2,2, sharex = True)
ax[0, 0].set_title("Energy")
ax[0, 1].set_title("Magnetization")
ax[1, 0].set_title("Specific heat")
ax[1, 1].set_title("Susceptibility")

# YOUR CODE HERE
pass

ax[0, 0].legend()
ax[0, 1].legend()
ax[1, 0].legend()
ax[1, 1].legend()


- [ ] Create an animation of the evolution of the system.

- [ ] There should be a correlation between samples. How will you compute it? or taking samples after a full sweep guarantees something?

- [ ] The statistic over several seeds can be computed in parallel using the multiprocessing module (for example). Implement it. 

- [ ] How does the computation time scales with $n$, for a full simulation?

- [ ] Is there a relationship between the number of samples and the system size?

- [ ] Use vpython to visualize the system and its evolution

- [ ] Investigate the Wang-Landau sampling. Implement it. (FINAL PROJECT) 